In [ ]:
# looking for reflected particles in phase space filaments

findex = 2 #9
step_choice = "last" #"first"

In [ ]:
# Imports and Folder Selection

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../v2_proc_files')
from useful_funcs import *
from TristanRun import TristanRun

%matplotlib widget

%load_ext autoreload
%autoreload 2 

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq

cc = run.cc
B0 = run.B0

tstep_choices = {"first": [run.steps[0]], "last": [run.steps[-1]], "all": run.steps}

my_steps = tstep_choices[step_choice]
boilerplate = run.boilerplate + f" tstep = {my_steps[0]}"


In [ ]:
# Config Data

d = run.config_data(my_steps, "both")

In [ ]:
print(run.stride)

In [ ]:
df = d.particles[1].to_dataframe()

ann_size(df)
add_gamma(df)

In [ ]:
roi = fq_cut(d.fields, 0, [0, run.xlim, 0, run.ylim]) 

In [ ]:
from matplotlib.path import Path

def choose_rect(df, rect):
    [x1, x2, y1, y2] = rect
    return df.loc[(x1 < df['x']) & (df['x'] < x2) & (y1 < df['y']) & (df['y'] < y2)]

def choose_circle(df, x0, y0, r):
    return df.loc[(df['x'] - x0) ** 2 + (df['y'] - y0) ** 2 < r ** 2]

def choose_circle_slice(df, x0, y0, r1, r2):
    df['dist'] = np.sqrt((df['x'] - x0) ** 2 + (df['y'] - y0) ** 2)

    return df.loc[(r1 < df['dist']) & (df['dist'] < r2)]

def choose_ellipse(df, x0, y0, a, b):
    return df.loc[((df['x'] - x0) / a) ** 2 + ((df['y'] - y0) / b) ** 2 < 1]

def choose_quad(df, points):

    # Create a Path object from the points
    path = Path(points)

    # Create an array of (x, y) points from the dataframe
    points_array = df[['x', 'y']].values

    # Use the Path object to determine which points are inside the quadrilateral
    mask = path.contains_points(points_array)

    # Return the dataframe rows that are inside the quadrilateral
    return df[mask]

In [ ]:
print(run.xlim / 2)

In [ ]:

chosen_sets = [
    choose_rect(df, [0, 1000, 2500, 5500])
]

#chosen_sets = [
#    choose_quad(df, [(337, 1255), (420, 1255), (417, 1380), (334, 1380)])
#]

chosen_sets = [
    choose_quad(df, [(320, 1255), (430, 1255), (417, 1380), (334, 1380)])
]

colors = ['blue', 'red', 'green', 'pink', 'yellow', 'purple', 'orange', 'cyan', 'magenta', 'black']


In [ ]:
quads = {9: [(123, 142), (126, 142), (132, 65), (128, 65)], 
         2: [(60, 180), (63, 180), (65, 220), (60, 220)]}

# 2 old: [(57, 180), (63, 180), (65, 220), (55, 220)]

particles = choose_quad(df, quads[findex])
print(len(particles))

var1 = 'u'
var2 = 'v'

plt.figure(figsize=(8, 5))
plt.scatter(particles[var1], particles[var2], s=1)
plt.xlabel(var1)
plt.ylabel(var2)
plt.title(f'{var2} vs {var1} for selected particles')
plt.tight_layout()
plt.show()

In [ ]:
# 2D histogram scatter plot with log density color map

x = particles[var1]
y = particles[var2]

# Compute 2D histogram
hist2d, xedges, yedges = np.histogram2d(x, y, bins=100)

# Get bin centers
xcenters = (xedges[:-1] + xedges[1:]) / 2
ycenters = (yedges[:-1] + yedges[1:]) / 2

# Plot
plt.figure(figsize=(8, 5))
plt.pcolormesh(xedges, yedges, np.log10(hist2d.T + 1), cmap='viridis')
plt.xlabel(var1)
plt.ylabel(var2)
plt.title(f'Log Density of {var2} vs {var1} for selected particles')
plt.colorbar(label='log10(density)')
plt.tight_layout()
plt.show()